In [1]:
import os
from pathlib import Path
from typing import Literal, Optional, List

import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import awkward as ak
from dataclasses import dataclass, field
import yaml

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec
import seaborn as sns

os.environ["PROJECT_PATH"] = "/home/syebes/lambda-ana"
project_path = Path(os.environ["PROJECT_PATH"])
sys.path.append(str(project_path))

# My imports
from src.driver import AnalysisDriver
import src.branches as branches
from src.cuts import CutConfig, create_cut

# Load files

In [2]:
# POT taken from preselection stage

lambda_pot = 1.428596135616784e+23
aurora_pot = 5.401777280854836e+20

In [3]:
# LOAD FRANS TREE

#lambda_file = 'root://fndcadoor.fnal.gov:1094/pnfs/fnal.gov/usr/sbnd//scratch/users/syebes/v10_14_02_04/ccqeLambda/OriginsAna_oldPandora/fransOutput_oldPandora.root'
lambda_file = 'root://fndcadoor.fnal.gov:1094/pnfs/fnal.gov/usr/sbnd//scratch/users/syebes/v10_14_02_04/ccqeLambda/OriginsAna_opPandora/fransOutput_opPandora.root'
#lambda_file = 'root://fndcadoor.fnal.gov:1094/pnfs/fnal.gov/usr/sbnd//scratch/users/syebes/v10_14_02_04/ccqeLambda/OriginsAna_cheat/fransOutput_cheatPandora.root'
aurora_file = "/exp/sbnd/data/users/syebes/eafData/fransOutputs/devFransOutput.root"

frans_tdir = "FRANSProducer/fransTree"
lambda_tree = uproot.open(lambda_file)[frans_tdir]
aurora_tree = uproot.open(aurora_file)[frans_tdir]


max_entry = None
indices =['run_id', 'subrun_id', 'event_id', 'slice_id']

lambda_df = pd.DataFrame(lambda_tree.arrays(branches.frans, library="np", entry_stop=max_entry))
aurora_df = pd.DataFrame(aurora_tree.arrays(branches.frans, library="np", entry_stop=max_entry))

lambda_df.set_index(indices, inplace=True)
aurora_df.set_index(indices, inplace=True)

# SCALING
pot_3y = 7e20

lambda_scale = pot_3y / lambda_pot
aurora_scale = pot_3y / aurora_pot

aurora_sgnl_mask = (aurora_df.n_lambda > 0) | (aurora_df.n_sigma0 > 0)

lambda_df['scale'] = pot_3y / (lambda_pot + aurora_pot)
aurora_df['scale'] = aurora_scale
aurora_df.loc[aurora_sgnl_mask, 'scale'] = pot_3y / (lambda_pot + aurora_pot)

df = pd.concat([lambda_df, aurora_df])

OSError: Failed to open file: [FATAL] Auth failed: No protocols left to try

# Michel tagging column

In [ ]:
'''
df["michel_final_score"] = np.where(
    df["michel_score"] == -1,
    2,
    df["michel_score"]
)'''

df["michel_final_score"] = np.where(
    ~df["michel_vars_computed"],
    2,
    df["michel_score"]
)

# Preselection

In [ ]:
preselected_mask = df.pre_pass

pre_df = df[preselected_mask]

# Selection

import importlib
import src

importlib.reload(src.driver)
importlib.reload(src.plotter)
importlib.reload(src.frans_branches)
importlib.reload(src.cuts)
from src.driver import AnalysisDriver
from src.frans_branches import branches
from src.cuts import CutConfig, create_cut

In [ ]:
with open("../../config/cuts_config.yaml", "r") as f:
    config = yaml.safe_load(f)

cuts_config = []

for cut_cfg in config['frans_cuts'].values():
    cfg = CutConfig(
        cut = create_cut(cut_cfg['cut']),
        **{k: v for k, v in cut_cfg.items() if k != "cut"}
    )

    cuts_config.append(cfg)

In [ ]:
driver = AnalysisDriver(pre_df, cuts_config)

driver.plot_cuts()
driver.get_cuts_performance(**config['initial_amounts'])
final_df = driver.get_final_df()

# Load FRANS PID tree

In [ ]:
pid_tdir = "lambdaPidPandoraAna/CalorimetryTree"

lambda_pid_tree = uproot.open(lambda_file)[pid_tdir]
aurora_pid_tree = uproot.open(aurora_file)[pid_tdir]

lambda_pid_df = pd.DataFrame(lambda_pid_tree.arrays(branches.frans_pid, library="np", entry_stop=max_entry))
aurora_pid_df = pd.DataFrame(aurora_pid_tree.arrays(branches.frans_pid, library="np", entry_stop=max_entry))

pid_df = pd.concat([lambda_pid_df, aurora_pid_df])
pid_df.set_index(['RunID', 'SubRunID', 'EventID'], inplace=True)

final_df.index = final_df.index.droplevel(-1)
selected_indices = final_df.index

pid_df = pid_df[pid_df.index.isin(selected_indices)]

cols = ['is_lambda_signal', 'scale', 'gen_index', 'n_hyperon', 'n_lambda', 'n_sigma0','nu_pdg',
 'nu_int_mode',
 'nu_int_type',
 'nu_cc_nc',]

pid_df[cols] = final_df.loc[selected_indices, cols].values

pid_df['pid_cut'] = (pid_df.NTracksHI >=1) & (pid_df.NTracksLI >=1)

In [ ]:
with open("../../config/cuts_config.yaml", "r") as f:
    config = yaml.safe_load(f)

pid_cuts_config = []

for cut_cfg in config['pid_cuts'].values():
    cfg = CutConfig(
        cut = create_cut(cut_cfg['cut']),
        **{k: v for k, v in cut_cfg.items() if k != "cut"}
    )

    pid_cuts_config.append(cfg)

In [ ]:
pid_driver = AnalysisDriver(pid_df, pid_cuts_config)

pid_driver.plot_cuts()
pid_driver.get_cuts_performance(**config['initial_amounts'])
final_pid_df = pid_driver.get_final_df()

In [ ]:
pid_driver.get_cuts_performance(**config['initial_amounts'])